In [114]:
from itertools import product

import argparse
from datasets import get_dataset
from ours_train_eval import *

from gib_gin import GIBGIN, Discriminator
from gib_gat import GIBGAT
from gib_sage import GIBSAGE
from gib_gcn  import GIBGCN
import numpy as np

import time

import torch
import torch.nn.functional as F
from torch import tensor
from torch.optim import Adam
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch_geometric.data import DataLoader, DenseDataLoader as DenseLoader
from torch_geometric.utils import subgraph
import numpy as np
import pickle


In [115]:
import torch
import torch.nn.functional as F
from torch_geometric.utils import subgraph, add_self_loops
from torch_geometric.data import Data,Batch
import tqdm
from collections import defaultdict
def calculate_fidelity(data, node_mask, model, remove_nodes=True, top_k=None):
    data = Batch.from_data_list([data])
    device = next(model.parameters()).device
    data = data.to(device)

    # Compute sparsity
    total_nodes = data.x.shape[0]
    sparsity = 1 - node_mask.sum().item() / total_nodes if total_nodes > 0 else 0

    # Get original predictions
    original_pred = model(data)[0]
    original_pred = F.softmax(original_pred, dim=1)
    label = original_pred.argmax(-1).item()
    
    # Apply node mask (InvFidelity)
    if remove_nodes:
        masked_edge_index, _ = subgraph(node_mask == 0, edge_index=data.edge_index, num_nodes=data.x.size(0), relabel_nodes=True)
        n_nodes = (node_mask == 0).sum()
        new_data = Batch.from_data_list([Data(x=data.x[node_mask == 0], edge_index=masked_edge_index)])
        masked_pred = model(new_data)[0]
    else:
        masked_x = data.x.clone()
        masked_x[node_mask==1] = 0
        new_data = Batch.from_data_list([Data(x=masked_x, edge_index=data.edge_index)])
        masked_pred = model(new_data)[0]
    masked_pred = F.softmax(masked_pred, dim=1)

    # # Keep only important nodes (Fidelity)
    # if remove_nodes:
    #     masked_edge_index, _ = subgraph(node_mask == 1, edge_index=data.edge_index, num_nodes=data.x.size(0), relabel_nodes=True)
    #     n_nodes = (node_mask == 1).sum()
    #     new_data = Batch.from_data_list([Data(x=data.x[node_mask == 1], edge_index=masked_edge_index)])
    #     retained_pred = model(new_data)[0]
    # else:
    #     masked_x = data.x.clone()
    #     masked_x[node_mask==0] = 0
    #     new_data = Batch.from_data_list([Data(x=masked_x, edge_index=data.edge_index)])
    #     retained_pred = model(new_data)[0]
    # retained_pred = F.softmax(retained_pred, dim=1)

    inv_fidelity = 0
    # Compute Fidelity+ and Fidelity-
    # inv_fidelity = (original_pred[:, label] - 
    #                              retained_pred[:, label]).mean().item()

    # fidelity = (original_pred[:, label] - 
    #                              masked_pred[:, label]).mean().item()

    # inv_fidelity = (original_pred.argmax(-1) != 
    #                              retained_pred.argmax(-1)).float().item()

    fidelity = (original_pred.argmax(-1)  != 
                                 masked_pred.argmax(-1) ).float().item()

    n_fidelity = inv_fidelity*sparsity
    n_inv_fidelity = inv_fidelity*(1-sparsity)
    
    # Compute HFidelity (harmonic mean of Fidelity+ and Fidelity-)
    hfidelity = ((1+n_fidelity) * (1-n_inv_fidelity)) / (2 + n_fidelity - n_inv_fidelity) if (1 + n_fidelity - n_inv_fidelity) != 0 else 0

    return {
        "Fidelity": fidelity,
        "InvFidelity": inv_fidelity,
        "HFidelity": hfidelity
    }

In [116]:
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d as BN
from torch_geometric.nn import GINConv, global_mean_pool, JumpingKnowledge
from torch_geometric.utils import to_dense_adj
def aggregate(self, assignment, x, batch, edge_index):

    max_id = torch.max(batch)
    if torch.cuda.is_available():
        EYE = torch.ones(2).cuda()
    else:
        EYE = torch.ones(2)

    all_adj = to_dense_adj(edge_index, max_num_nodes=x.shape[0])[0]

    all_pos_penalty = 0
    all_graph_embedding = []
    all_pos_embedding = []

    st = 0
    end = 0

    for i in range(int(max_id + 1)):

        j = 0
        while batch[st + j] == i and st + j <= len(batch) - 2:
            j += 1

        end = st + j

        if end == len(batch) - 1:
            end += 1

        one_batch_x = x[st:end]
        one_batch_assignment = assignment[st:end]

        group_features = torch.mm(torch.t(one_batch_assignment), one_batch_x)

        pos_embedding = group_features[0].unsqueeze(dim=0)

        Adj = all_adj[st:end,st:end]
        new_adj = torch.mm(torch.t(one_batch_assignment), Adj)
        new_adj = torch.mm(new_adj, one_batch_assignment)
        normalize_new_adj = F.normalize(new_adj, p=1, dim=1)
        norm_diag = torch.diag(normalize_new_adj)
        pos_penalty = self.mse_loss(norm_diag, EYE)
        graph_embedding = torch.mean(x, dim=0, keepdim=True)

        all_pos_embedding.append(pos_embedding)
        all_graph_embedding.append(graph_embedding)

        all_pos_penalty = all_pos_penalty + pos_penalty

        st = end

    all_pos_embedding = torch.cat(tuple(all_pos_embedding), dim=0)
    all_graph_embedding = torch.cat(tuple(all_graph_embedding), dim=0)
    all_pos_penalty = all_pos_penalty / (max_id + 1)

    return all_pos_embedding,all_graph_embedding, all_pos_penalty


In [117]:
# def generate_hard_masks(soft_mask):
#     sparsity_levels = torch.arange(0.5,1, 0.05)
#     hard_masks = []
#     for sparsity in sparsity_levels:
#         threshold = np.percentile(soft_mask, sparsity * 100)
#         hard_mask = (soft_mask > threshold).int()
#         if hard_mask.sum() == 0:
#             hard_mask = (soft_mask > soft_mask.min()).int()
#         hard_masks.append(hard_mask)
#     return list(zip(sparsity_levels, hard_masks))
import torch
import numpy as np

import torch

def generate_hard_masks(soft_mask):
    soft_mask_flat = soft_mask.flatten()
    total_elements = soft_mask_flat.numel()
    sparsity_levels = torch.arange(0.5, 1.0, 0.05)
    hard_masks = []

    # Get sorted indices (ascending: lowest values first)
    sorted_indices = torch.argsort(soft_mask_flat)

    for sparsity in sparsity_levels:
        num_to_mask = int(sparsity.item() * total_elements)
        mask_flat = torch.ones_like(soft_mask_flat, dtype=torch.int)
        
        if num_to_mask >= total_elements:
            num_to_mask = total_elements - 1  # keep at least one element
        
        # Zero out the lowest `num_to_mask` elements
        mask_flat[sorted_indices[:num_to_mask]] = 0
        
        # Reshape to original shape
        hard_mask = mask_flat.view_as(soft_mask)
        hard_masks.append(hard_mask)

    return list(zip(sparsity_levels, hard_masks))



In [126]:
dataset_name = 'BBBP'
seeds_results = pickle.load(open(f'results/{dataset_name}.pkl', 'rb'))

FileNotFoundError: [Errno 2] No such file or directory: 'results/bbbp.pkl'

In [122]:
import pandas as pd
l = []
for seed in range(len(seeds_results)):
    train_idx, val_idx, test_idx, model = seeds_results[seed][0]
    model.aggregate = lambda *x: aggregate(model, *x)
    model = model.to(device)
    dataset = get_dataset(dataset_name, sparse=True)
    batch_size = 1
    train_dataset = dataset[train_idx]
    test_dataset = dataset[test_idx]
    val_dataset = dataset[val_idx]
    
    if 'adj' in train_dataset[0]:
        train_loader = DenseLoader(train_dataset, batch_size, shuffle=True)
        val_loader = DenseLoader(val_dataset, batch_size, shuffle=False)
        test_loader = DenseLoader(test_dataset, batch_size, shuffle=False)
    else:
        train_loader = DataLoader(train_dataset, batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size, shuffle=False)
    acc = eval_acc(model, test_loader)
    seed_expl_res = []
    for data in tqdm.tqdm(test_loader):
        if data.y == 0: continue
        try:
            out, _, _, _, assignment = model(data.to(device), with_assignment=True)
            c = out.argmax(-1).item()
            masks = generate_hard_masks(assignment[:,c].cpu().detach())
            d = {}
            for m in masks:
                d[m[0].item()] = calculate_fidelity(data, m[1], model)
            seed_expl_res.append(pd.DataFrame(d))
        except Exception as e:
            print(e)
            continue
    l.append(sum(seed_expl_res)/len(seed_expl_res))

 30%|██▉       | 33/111 [00:01<00:04, 16.39it/s]

max(): Expected reduction dim to be specified for input.numel() == 0. Specify the reduction dim with the 'dim' argument.


 77%|███████▋  | 85/111 [00:04<00:01, 20.46it/s]

max(): Expected reduction dim to be specified for input.numel() == 0. Specify the reduction dim with the 'dim' argument.


  6%|▋         | 7/111 [00:00<00:06, 15.52it/s]

...

 57%|█████▋    | 63/111 [00:03<00:02, 23.84it/s]

max(): Expected reduction dim to be specified for input.numel() == 0. Specify the reduction dim with the 'dim' argument.


 25%|██▌       | 28/111 [00:00<00:02, 29.75it/s]

max(): Expected reduction dim to be specified for input.numel() == 0. Specify the reduction dim with the 'dim' argument.


100%|██████████| 111/111 [00:05<00:00, 19.77it/s]


In [123]:
l = [np.array(x) for x in l]

In [124]:
np.mean(l, axis=0)[0]

array([0.39599591, 0.37936604, 0.38892924, 0.37779093, 0.34179498,
       0.33154018, 0.30713466, 0.27391674, 0.25240666, 0.23323247])

In [72]:
np.std(l, axis=0)[0]

array([0.09983972, 0.1066425 , 0.11667583, 0.11883211, 0.11872186,
       0.12351298, 0.13286887, 0.14500764, 0.13102667, 0.09104237])

In [25]:
sum(l)/len(l)

,0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90,0.95
Fidelity,0.365079,0.323413,0.292063,0.324206,0.248413,0.327778,0.274206,0.217063,0.18373,0.101587
InvFidelity,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
HFidelity,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.50000,0.500000


In [ ]:
import torch
from torch_geometric.data import Data, Batch
import networkx as nx
import matplotlib.pyplot as plt
import math
def plot_activations(batch_ids, batch, attr):
    if type(batch_ids) != list:
        batch_ids = [batch_ids]
    num_ids = len(batch_ids)
    cols = 5
    rows = math.ceil(num_ids / cols)
    
    fig, axs = plt.subplots(rows, cols, figsize=(16*5, 8 * rows))
    
    if type(axs) != np.ndarray: axs = np.array([axs])
    # Flatten axs if it's 2D to simplify indexing
    axs = axs.flatten()

    for i, batch_id in enumerate(batch_ids):
        node_mask = batch.batch == batch_id  # Get nodes where batch == 0
        if node_mask.float().sum() == 0: continue
        node_indices = torch.nonzero(node_mask, as_tuple=True)[0]
        
        subgraph_edge_mask = (batch.batch[batch.edge_index[0]] == batch_id) & \
                             (batch.batch[batch.edge_index[1]] == batch_id)
        subgraph_edges = batch.edge_index[:, subgraph_edge_mask]
        
        node_mapping = {old_idx.item(): new_idx for new_idx, old_idx in enumerate(node_indices)}
        remapped_edges = torch.tensor([[node_mapping[e.item()] for e in edge] for edge in subgraph_edges.T])
        
        G = nx.Graph()
        G.add_edges_from(remapped_edges.numpy())
        
        nx.set_node_attributes(G, {v: k for k, v in node_mapping.items()}, "original_id")
        
        node_colors = []
        node_borders = []
        
        for node in G.nodes:
            if attr[batch.batch==batch_id][node] == 1:
                node_colors.append("lightblue")  # Fill color
                node_borders.append("red")  # Border color for attr == 1
            else:
                node_colors.append("lightblue")  # Fill color
                node_borders.append("black")  # Default border color
        
        
        pos = nx.kamada_kawai_layout(G) 
        
        nx.draw(
            G, pos,
            node_color=node_colors,
            edgecolors=node_borders,  # Border colors
            node_size=700,
            with_labels=True,
            ax = axs[i]
        )
        
        axs[i].set_title(f"Class = {batch.y[batch_id]}")
    plt.show()


In [ ]:
data = dataset[499]

In [ ]:
pred, _, _, _, assignment = model(Batch.from_data_list([data]).to(device), with_assignment=True)
print(pred)

In [ ]:
c = pred.argmax(-1).item()
print(c)
node_mask = generate_hard_masks(assignment[:,c])[3][1]
plot_activations([0], Batch.from_data_list([data]), node_mask)

In [ ]:
# masked_edge_index, _ = subgraph(node_mask == 0, edge_index=data.edge_index, num_nodes=data.x.size(0), relabel_nodes=True)
# n_nodes = (node_mask == 0).sum()
# new_data = Batch.from_data_list([Data(x=data.x[node_mask == 0], edge_index=masked_edge_index, num_nodes=n_nodes)[0])])



edge_mask = (~node_mask.bool()[data.edge_index[0]]) & (~node_mask.bool()[data.edge_index[1]])
new_data = Batch.from_data_list([Data(x=data.x, edge_index=data.edge_index[:,edge_mask], y=0)])
masked_pred = model(new_data.to(device))[0]

In [ ]:
masked_pred

In [ ]:
plot_activations([0], Batch.from_data_list([new_data]), torch.ones(new_data.x.shape[0]))

In [ ]:
new_data.x.shape